In [1]:
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, KFold, cross_validate
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from xgboost import XGBRegressor

In [2]:
df = pd.read_excel("Tension.xlsx")
df = df.drop(columns=['Mixture', 'Specimen'])
df = df.dropna(subset=['Fiber Type'])
df = df.reset_index(drop=True)

In [3]:
df['has_Silica_Fume']        = (df['Silica Fume'] > 0).astype(int)
df['FlyAshF_x_SilicaFume']  = df['Fly ash F'] * df['Silica Fume']
df['Length_x_SilicaFume']   = df['Length (mm)'] * df['Silica Fume']
df['FiberVol_x_Length']     = df['Fiber Volume'] * df['Length (mm)']
df['FlyAshF_x_WaterReducer']= df['Fly ash F'] * df['Water Reducer / SP']
df['Total_SCM']             = df['Fly ash C'] + df['Fly ash F'] + df['GGBS'] + df['Silica Fume']
df['SCM_ratio']             = df['Total_SCM'] / df['Cement']
df['Total_paste']           = df['Cement'] + df['Water'] + df['Total_SCM']
df['Water_Binder']          = df['Water'] / (df['Cement'] + df['Total_SCM'])

df = pd.get_dummies(df, columns=['Fiber Type'], drop_first=False)
print(f"Final shape: {df.shape}")

Final shape: (503, 53)


In [4]:
X= df.drop(columns=['Second Stress', 'Second Strain'])
y_stress = df['Second Stress']
y_strain = df['Second Strain']

In [5]:
X_train, X_test, \
y_stress_train, y_stress_test, \
y_strain_train, y_strain_test = train_test_split(
    X, y_stress, y_strain,
    test_size=0.20,
    random_state=42
)

In [6]:
scaler = StandardScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train),
                               columns=X.columns, index=X_train.index)
X_test_scaled  = pd.DataFrame(scaler.transform(X_test),
                               columns=X.columns, index=X_test.index)

kf = KFold(n_splits=5, shuffle=True, random_state=42)

print(f"Train: {X_train_scaled.shape[0]} rows | Test: {X_test_scaled.shape[0]} rows | Features: {X_train_scaled.shape[1]}")

Train: 402 rows | Test: 101 rows | Features: 51


In [7]:
print("\n" + "="*60)
print("  STAGE 1 — STRESS MODELS")
print("="*60)

stress_models = {
    "RF":  RandomForestRegressor(random_state=42),
    "XGB": XGBRegressor(random_state=42, objective='reg:squarederror', verbosity=0),
}

fitted_stress_models = {}
for name, model in stress_models.items():
    cv = cross_validate(model, X_train_scaled, y_stress_train, cv=kf,
                        scoring=('r2', 'neg_root_mean_squared_error'))
    r2   = cv['test_r2'].mean()
    rmse = -cv['test_neg_root_mean_squared_error'].mean()
    print(f"  {name:5s}  CV R²={r2:.4f}  CV RMSE={rmse:.4f}")

    # Fit on full training set for Stage 2
    model.fit(X_train_scaled, y_stress_train)
    fitted_stress_models[name] = model


  STAGE 1 — STRESS MODELS
  RF     CV R²=0.8215  CV RMSE=0.6408
  XGB    CV R²=0.8351  CV RMSE=0.6120


In [8]:
print("\n" + "="*60)
print("  STAGE 2 — CHAINED STRAIN MODELS  (predicted stress as feature)")
print("="*60)

def oof_stress_predictions(stress_model_cls, stress_model_params,
                           X_tr, y_stress_tr):
    """Generate out-of-fold stress predictions for training rows."""
    oof_preds = np.zeros(len(X_tr))
    for fold_train_idx, fold_val_idx in kf.split(X_tr):
        m = stress_model_cls(**stress_model_params)
        m.fit(X_tr.iloc[fold_train_idx], y_stress_tr.iloc[fold_train_idx])
        oof_preds[fold_val_idx] = m.predict(X_tr.iloc[fold_val_idx])
    return oof_preds


strain_results_chained = []

for sname, stress_model in fitted_stress_models.items():

    # --- Build augmented train set with OOF stress predictions ---
    if sname == "RF":
        oof_stress = oof_stress_predictions(
            RandomForestRegressor, {"random_state": 42},
            X_train_scaled, y_stress_train
        )
    else:
        oof_stress = oof_stress_predictions(
            XGBRegressor,
            {"random_state": 42, "objective": "reg:squarederror", "verbosity": 0},
            X_train_scaled, y_stress_train
        )

    X_train_aug = X_train_scaled.copy()
    X_train_aug['predicted_stress'] = oof_stress

    # --- Build augmented test set with direct stress predictions ---
    test_stress_pred = stress_model.predict(X_test_scaled)
    X_test_aug = X_test_scaled.copy()
    X_test_aug['predicted_stress'] = test_stress_pred

    print(f"\n  Stress provider: {sname}")

    for strname, strain_model_cls, params in [
        ("RF",  RandomForestRegressor, {"random_state": 42}),
        ("XGB", XGBRegressor,          {"random_state": 42, "objective": "reg:squarederror", "verbosity": 0}),
    ]:
        # CV on augmented training data
        strain_model = strain_model_cls(**params)
        cv = cross_validate(strain_model, X_train_aug, y_strain_train, cv=kf,
                            scoring=('r2', 'neg_root_mean_squared_error'))
        cv_r2   = cv['test_r2'].mean()
        cv_rmse = -cv['test_neg_root_mean_squared_error'].mean()

        # Test set
        strain_model.fit(X_train_aug, y_strain_train)
        pred_strain = strain_model.predict(X_test_aug)
        test_r2   = r2_score(y_strain_test, pred_strain)
        test_rmse = np.sqrt(mean_squared_error(y_strain_test, pred_strain))
        test_mae  = mean_absolute_error(y_strain_test, pred_strain)

        label = f"Stress({sname}) → Strain({strname})"
        print(f"    {label:35s}  CV R²={cv_r2:.4f}  CV RMSE={cv_rmse:.6f} | "
              f"Test R²={test_r2:.4f}  RMSE={test_rmse:.4f}  MAE={test_mae:.4f}")

        strain_results_chained.append({
            "Stress Model": sname, "Strain Model": strname,
            "CV R2": round(cv_r2, 4), "CV RMSE": round(cv_rmse, 6),
            "Test R2": round(test_r2, 4), "Test RMSE": round(test_rmse, 4),
            "Test MAE": round(test_mae, 4)
        })


  STAGE 2 — CHAINED STRAIN MODELS  (predicted stress as feature)

  Stress provider: RF
    Stress(RF) → Strain(RF)              CV R²=0.4772  CV RMSE=0.013631 | Test R²=0.6975  RMSE=0.0130  MAE=0.0097
    Stress(RF) → Strain(XGB)             CV R²=0.4392  CV RMSE=0.014107 | Test R²=0.6773  RMSE=0.0134  MAE=0.0097

  Stress provider: XGB
    Stress(XGB) → Strain(RF)             CV R²=0.4584  CV RMSE=0.013869 | Test R²=0.7133  RMSE=0.0126  MAE=0.0093
    Stress(XGB) → Strain(XGB)            CV R²=0.4193  CV RMSE=0.014305 | Test R²=0.6768  RMSE=0.0134  MAE=0.0097


In [9]:
print("\n" + "="*60)
print("  BASELINE vs CHAINED — STRAIN COMPARISON")
print("="*60)

baseline = {
    "RF":  {"Test R2": 0.7450, "Test RMSE": 0.0119, "Test MAE": 0.0089},
    "XGB": {"Test R2": 0.7394, "Test RMSE": 0.0121, "Test MAE": 0.0090},
}

print(f"\n{'Model':<40} {'Test R²':>9} {'RMSE':>10} {'MAE':>10}")
print("-"*70)

for k, v in baseline.items():
    print(f"  Baseline Strain({k}){'':<20} {v['Test R2']:>9.4f} {v['Test RMSE']:>10.4f} {v['Test MAE']:>10.4f}")

print()
for r in strain_results_chained:
    label = f"  Chained Stress({r['Stress Model']}) → Strain({r['Strain Model']})"
    print(f"{label:<40} {r['Test R2']:>9.4f} {r['Test RMSE']:>10.4f} {r['Test MAE']:>10.4f}")

print("\n✅ Chained pipeline complete.")


  BASELINE vs CHAINED — STRAIN COMPARISON

Model                                      Test R²       RMSE        MAE
----------------------------------------------------------------------
  Baseline Strain(RF)                        0.7450     0.0119     0.0089
  Baseline Strain(XGB)                        0.7394     0.0121     0.0090

  Chained Stress(RF) → Strain(RF)           0.6975     0.0130     0.0097
  Chained Stress(RF) → Strain(XGB)          0.6773     0.0134     0.0097
  Chained Stress(XGB) → Strain(RF)          0.7133     0.0126     0.0093
  Chained Stress(XGB) → Strain(XGB)         0.6768     0.0134     0.0097

✅ Chained pipeline complete.
